In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import pandas as pd

from src.prediction.hotspot_predictor import HotspotPredictor
from src.prediction.geofence_generator import GeofenceGenerator

In [2]:
processed_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

df = pd.read_pickle(
    processed_path
    / "final_pci_dataset.pkl"
)

print(df.shape)

(298450, 35)


In [3]:
features = [

    "hour",

    "month",

    "road_criticality",

    "hotspot_score",

    "violation_score",

    "temporal_score",

    "pci"
]

X = df[features]

y = df["hotspot_score"]

In [4]:
predictor = HotspotPredictor()

predictor.train(
    X,
    y
)

MAE: 0.0000


In [5]:
df["predicted_risk"] = (
    predictor.predict(X)
)

df["predicted_risk"].describe()

count    298450.000000
mean          0.288879
std           0.367087
min           0.000454
25%           0.021706
50%           0.096032
75%           0.448802
max           1.000000
Name: predicted_risk, dtype: float64

In [6]:
high_risk = GeofenceGenerator.generate(
    df,
    threshold=0.80
)

print(high_risk.shape)

(56252, 36)


In [7]:
top_zones = (

    high_risk

    .groupby("location")

    .agg(
        mean_risk=(
            "predicted_risk",
            "mean"
        ),

        count=(
            "id",
            "count"
        )
    )

    .sort_values(
        "mean_risk",
        ascending=False
    )
)

top_zones.head(20)

,mean_risk,count
location,,
"1st Cross Road, Anchepete, Chickpete, Bengaluru, Karnataka. Pin-560002 (India)",1.0,4
"Nisar Metals, 175/1, B.V.K. Iyengar Road, Balepet, Chikkapete, Cottonpete, Bengaluru Central City Corporation, Bengaluru, Bangalore North, Bengaluru Urban, Karnataka, 560053, India",1.0,4
"Narasimha Raj Wadiyar Road, Gollarpet, Nagartapete, Bengaluru, Karnataka. Pin-560002 (India)",1.0,58
"Narasimha Raj Wadiyar Road, New Bamboo Bazar, Kalasipalyam, Bengaluru, Karnataka. Pin-560002 (India)",1.0,107
"Narasimha Raj Wadiyar Road, Potters Colony, Kalasipalyam, Bengaluru, Karnataka. Pin-560002 (India)",1.0,110
"New Bombay Bazar Road, City Market Circle, Bengaluru, Karnataka. Pin-560002 (India)",1.0,15
"New Bombay Bazar Road, New Bamboo Bazar, Kalasipalyam, Bengaluru, Karnataka. Pin-560002 (India)",1.0,3
"New Shanthi Sagar, 8th Cross Road, Balepet, Chikkapete, Chickpete, Bengaluru Central City Corporation, Bengaluru, Bangalore North, Bengaluru Urban, Karnataka, 560053, India",1.0,1
"OTC Road, Majestic Circle, Majestic, Bengaluru, Karnataka. Pin-560023 (India)",1.0,1
